# Build CLIP Model and Train It on Fashion MNIST Dataset

<pre>
Source:
    https://medium.com/correll-lab/building-clip-from-scratch-68f6e42d35f4

Adapted:
    Antonio Esteves @ UMinho, May 2025
</pre>

---

Computer vision models were historically limited to a fixed set of classes, but CLIP has been a revolution allowing open world object recognition by “predicting which image and text pairings go together". CLIP is able to predict this by learning the cosine similarity between image and text feature for batches of training data. This is shown in the contrastive pre-training portion of Figure 1 where the dot product between the image features ${I_1, ...,I_N}$ and the text features ${T_1,..., T_N}$ is taken.

In this tutorial, we are going to build CLIP from scratch and test it on the fashion MNIST dataset. Some of the sections in this article are taken from my vision transformers article, which will help you to understand how transformer models can be applied to images.

![Figure 1: CLIP Model Overview.](../fig/clip_model_01A.jpg)

## Import Libraries and Modules

We are going to be building CLIP using PyTorch. We also need to import in `torchvision.transforms` in order to resize the input images and convert them to tensors. Resizing the input images is optional, we just need to make sure that the image size is divisible by the patch size. We are going to be using Adam for our optimizer, so we need to import it in from `torch.optim`. We are going to import the fashion MNIST dataset from Hugging Face, so we need to import `datasets`. We are going to use the `DataLoader` from PyTorch to help load the data during training, so we need to import that it too.

In [ ]:
import os
import wandb
import torch
import ml_collections
import torch.nn               as     nn
import torch.optim            as     optim
import torchvision.transforms as     T
from   torch.utils.data       import Dataset, DataLoader
from   datasets               import load_dataset
import matplotlib.pyplot      as     plt
import numpy                  as     np
from   tqdm.notebook          import tqdm
from   tqdm                   import trange

In [ ]:
BOS       = 2 # Begin of (text) Sequence
EOS       = 3 # End of (text) Sequence
PAD_TOKEN = 0 # Text sequence padding value

## Image and Text Encoders

We will first build the image and text encoders. Both embed images and text, respectively, into a single token, which can then be used in the contrastive loss computation.

### Positional Embedding

![Figure 2: Image patches identification to preserve image structure.](../fig/clip_model_01B.jpg)

Unlike models such as LSTMs that get input embeddings sequentially, transformers get embeddings in parallel. While this increases the speed, transformers are not aware of the of the order in a sequences. This is a problem because changing the order of the sequence would most likely alter its structure. An example of this is Figure 2, which shows that changing an image’s patch order can change the image from an circle to something different. In order to fix this problem, positional encodings need to be added to the embeddings. Each positional encoding is unique to the position that it represents in a sequence, which allows the model to identify which position each embedding is supposed to represent. In order for the positional encodings to be added to the embeddings, they have to have the same dimension, `d_model`. We calculate the positional encodings with the next equations.

$PE_{(pos,2i)} = sin(\frac{pos}{10000^{2i/d_{model}}})$ <BR>
$PE_{(pos,2i+1)} = cos(\frac{pos}{10000^{2i/d_{model}}})$ <BR>

Notice that `pe` is only a local variable and only gets added to the class using the `register_buffer` method. This way, the positional encoding become a non-trainable parameter of the model.

In the `forward` method, the positional encodings that we calculated are added to the input `x`.

In [ ]:
class PositionalEmbedding(nn.Module):
  def __init__(self, width, max_seq_length):
    super().__init__()

    # Creating positional encoding
    pe = torch.zeros(max_seq_length, width)

    for pos in range(max_seq_length):
      for i in range(width):
        if i % 2 == 0:
          pe[pos][i] = np.sin(pos/(10000 ** (i/width)))
        else:
          pe[pos][i] = np.cos(pos/(10000 ** ((i-1)/width)))

    self.register_buffer('pe', pe.unsqueeze(0))

  def forward(self, x):
    # Add positional encoding to embeddings
    x = x + self.pe

    return x

## Attention Head

![Figure 3: Scaled Dot-Product Attention and Multi-Head Attention diagrams.](../fig/clip_model_01C.jpg)

Transformers use attention to which parts in the input sequence are more related with any other part. Attention scores can be calculated using the next equation.

$Attention(Q,K,V) = softmax(\frac{Q.K^T}{\sqrt{d_k}}).V$

The first step in calculating attention is obtaining the queries, keys, and values from the input tokens. The query of a token is what the token is looking for, the key is what the token contains, and the value is what is communicated between the tokens. The queries, keys, and values can be calculated by passing tokens through linear layers.

```python
def forward(self, x):
   # Obtaining queries, keys, and values
   Q = self.query(x)
   K = self.key(x)
   V = self.value(x)
```
We measure the relationship between tokens in tyhe input sequence by calculating the dot product between the queries and keys.

```python
# Dot product of queries and keys
attention = Q @ K.transpose(-2,-1)
```
We need to scale the dot product values to control variance at initialization, so that tokens are able to aggregate information from more than one token. Scaling is obtained by dividing the dot product by the square root of the size of the keys.

```python
# Scaling the dot product
attention = attention / (self.head_size ** 0.5)
```
The main difference between the transformer encoder and decoder is that decoder applies an attention mask while the encoder do not. While CLIP is an encoder only model, a mask still needs to be applied with the text encoder due to the padding that is applied to the input text during tokenization. Note that the mask is optional, so this Attention head can be used in both the text and image encoders. Masking consists in assigning $-\infty$ to the padding tokens, that is, those equal zero. 

![Figure 6: Example of applying mask to attention scores.](../fig/clip_model_01D.jpg)

```python
# Applying the attention mask
if mask is not None:
    attention = attention.masked_fill(mask == 0, float("-inf"))
```
We then need to apply a `softmax` operation on the scaled dot product. Here, values with negative infinity will simply be ignored.

```python
attention = torch.softmax(attention, dim=-1)
```
Finally, we need to get the dot product between the `softmax`output and the values matrix. This is essentially communicating the information between the corresponding tokens.

In [ ]:
class AttentionHead(nn.Module):
  def __init__(self, width, head_size):
    super().__init__()
    self.head_size = head_size

    self.query = nn.Linear(width, head_size)
    self.key   = nn.Linear(width, head_size)
    self.value = nn.Linear(width, head_size)

  def forward(self, x, mask=None):
    # Obtain queries, keys, and values
    Q = self.query(x)
    K = self.key(x)
    V = self.value(x)

    # Calculate the dot product between queries and keys
    attention = Q @ K.transpose(-2,-1)

    # Scaling the dot product
    attention = attention / (self.head_size ** 0.5)

    # Applying the attention mask
    if mask is not None:
        attention = attention.masked_fill(mask == 0, float("-inf"))

    # Obtaining attention scores as probabilities
    attention = torch.softmax(attention, dim=-1)

    # Multiply the probabilities with the values
    attention = attention @ V

    return attention

## Multi-Head Attention

Multi-head attention is just calculating multiple heads of self-attention in parallel and combining them. We can do this by adding the attention heads into a module list.

```python
self.heads = nn.ModuleList(
    [
    AttentionHead(width, self.head_size) for _ in range(n_heads)
    ]
)
```
Then, in the forward pass we feed all heads with the input tokens and concatenate their results.

```python
def forward(self, x, mask=None):
    # Concatenate the attention heads
    out = torch.cat([head(x, mask=mask) for head in self.heads], dim=-1)
```

In the end, we pass the concatenation output through a linear layer (weights $W_o$).

```python
out = self.W_o(out)
return out
```

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, width, n_heads):
    super().__init__()
    self.head_size = width // n_heads

    # Linear layer at the output
    self.W_o = nn.Linear(width, width)

    self.heads = nn.ModuleList(
        [
            AttentionHead(width, self.head_size) for _ in range(n_heads)
        ]
    )

  def forward(self, x, mask=None):
    # Concatenate the attention heads
    out = torch.cat([head(x, mask=mask) for head in self.heads], dim=-1)

    out = self.W_o(out)

    return out

## Transformer Encoder

![Figure 7: Transformer Encoder Diagram. Image: Own work.](../fig/clip_model_01E.jpg)

The transformer encoder is made up of four layers: the first layer applies normalization, the second layer is the multi-head attention, the third layer is another normalization layer, and the fourth one is a multi-layer perceptron.

Normalization is a regularization technique that normalizes the inputs in the batch by applying the following expression, where $\mathbb{E}[x]$ is the mean and $Var(x)$ is the variance of $x$.

$y = \frac{x−\mathbb{E}[x]}{Var[x]+\epsilon} * \gamma + \beta$

```python
# First normalization layer
self.ln1 = nn.LayerNorm(width)

# Second normalization layer
self.ln2 = nn.LayerNorm(width)
```

The MLP consists of two linear layers with a GELU activation in between. GELU is used instead of RELU because it does not have RELU’s limitation of being non-differentiable at zero.

```python
# Multi-layer perceptron
self.mlp = nn.Sequential(
    nn.Linear(width, width*r_mlp),
    nn.GELU(),
    nn.Linear(width*r_mlp, width)
)
```

In the `forward` method of the encoder, the input `x` is passed through the first normalization layer, the the output is fed into the multi-head attention. There is a residual connection that adds the original input to the output of the multi-head attention.

The output of the addition goes through the second normalization layer and the normalized output is applied into the MLP. After that, there is another residual connection that adds the input with the MLP output.

The residual connections help to prevent the vanishing gradient problem by creating a path for the gradient to be backpropagated unimpeded in direction to the beginning of network.

```python
def forward(self, x):
    # Residual Connection After Sub-Layer 1
    out = x + self.mha(self.ln1(x))

    # Residual Connection After Sub-Layer 2
    out = out + self.mlp(self.ln2(out))

    return out
```

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, width, n_heads, r_mlp=4):
        super().__init__()
        self.width = width
        self.n_heads = n_heads

        # First normalization layer
        self.ln1 = nn.LayerNorm(width)

        # Multi-head attention
        self.mha = MultiHeadAttention(width, n_heads)

        # First normalization layer
        self.ln2 = nn.LayerNorm(width)

        # Multi-layer perceptron
        self.mlp = nn.Sequential(
            nn.Linear(self.width, self.width*r_mlp),
            nn.GELU(),
            nn.Linear(self.width*r_mlp, self.width)
        )


    def forward(self, x, mask=None):
        # Residual connection after the norm-MHA block
        x = x + self.mha(self.ln1(x), mask=mask)

        # Residual connection after the norm-MLP block
        x = x + self.mlp(self.ln2(x))

        return x

## Tokenizer

Transformers are unable to process raw text, so the first thing we need to do is tokenize the input string before passing them through the text encoder.

Here, we are going to implement a simple tokenization where we just use the UTF-8 encoding. We can resort to the UTF-8 encoding for tokenization because we will deal with simple text in our examples. For more complex examples, we may use a [Byte-Pair Encoding](https://huggingface.co/learn/llm-course/chapter6/5) (BPE) tokenizer. With UTF encoding, the maximum vocabulary size is 256, which means that in complex examples we may have longer input sequences which would be inefficient when doing attention due to the limited context length.

![Figure 8: Tokenization Process with Max Sequence Length of 10.](../fig/clip_model_01F.jpg)

The first step of the tokenization is adding the begin of sequence (code 2) and end of sequence (code 3) tokens to the input string.

```python
text = chr(BOS) + text + chr(EOS)
```

After adding the BOS and EOS text tokens, we need to adjust (padding) the length of the sequence to the maximum sequence length (10 in this example).

```python
text = text + "".join([chr(PAD_TOKEN) for _ in range(10-len(text))])
```

We complete the tokenization by encoding the text sequence to UTF-8 and converting the output to an integer tensor (`IntTensor`).

```python
text = torch.IntTensor(list(text.encode("utf-8")))
```

After tokenizing the text, we need to create a mask for the text. While the mask that is normally used in transformers is used to ensure that tokens do not communicate with future tokens, the mask that we are applying here is just to ensure that padding tokens are ignored. Because of this, the mask is just going to be a tensor of size equal to the maximum sequence length, where each element of the mask is 0 when the corresponding token is padding and is 1 otherwise.

```python
mask = torch.ones(len(text.nonzero()))
mask = torch.cat((mask,torch.zeros(10-len(mask)))).type(torch.IntTensor)
```

In [ ]:
def tokenizer(text, encode=True, mask=None, max_seq_length=32):
    if encode:
        out  = chr(BOS) + text + chr(EOS) # Add BOS and EOS tokens
        out  = out + "".join([chr(PAD_TOKEN) for _ in range(max_seq_length-len(out))]) # Adding padding
        out  = torch.IntTensor(list(out.encode("utf-8"))) # Encode text
        mask = torch.ones(len(out.nonzero()))
        mask = torch.cat((mask,torch.zeros(max_seq_length-len(mask)))).type(torch.IntTensor)
    else:
        out  = [chr(x) for x in text[1:len(mask.nonzero())-1]]
        out  = "".join(out)
        mask = None

    return out, mask

## Text Encoder

For text encoder, we are going to use a regular transformer model. The first step in creating the text encoder is creating an embedding table of size (`vocab_size`, `width`). This embedding table implements a vector representation with `vocab_size` entries and each entry is a token's vector representation of size equal to the width of the transformer model (`width`).

```python
self.encoder_embedding = nn.Embedding(vocab_size, width)
```
Before outputting the result of the text encoder, we have to project the text features into a joint embedding space. We are going to do this by calculating the dot product between the text features and the learned projection weights, which we create by using a PyTorch `nn.Parameter`.

```python
# Learned projection of token's vector representation to joint embedding space
self.projection = nn.Parameter(torch.randn(width, emb_dim))
```

In the forward method, the first thing to do is pass the text tokens through the embedding layer.

```python
# Obtain the token's vector representation (token<=>character)
x = self.encoder_embedding(text)
```

We then need to add the positional encodings to the output of the embedding table.

```python
# Add the the positional embedding to each token vector representation
x = self.positional_embedding(x)
```

With the positional encodings added, we can now pass the tokens' vector representation through the transformer blocks of the text encoder along with the masks.

```python
# Pass the tokens+PE and masks through the transformer blocks of the text encoder
for encoder_layer in self.encoder:
    x = encoder_layer(x, mask=mask)
```

The output of the transformer blocks text encoder is the text features produced after applying the EOS at the input. If we opt for another text encoder, such as the BERT model, we use as output of the encoder, the output produced when we apply the class token at the input.

```python
# Use as encoded representation the text features produced for the EOS token
x = x[torch.arange(text.shape[0]),torch.sub(torch.sum(mask[:,0],dim=1),1)]
```

Finally, we project the encoded text features into the joint embedding space by calculating the dot product between the encoded features and the learned projection, and we normalize the result dividing by its norm.

```python
# Project encoded text features into the joint embedding space
if self.projection is not None:
    x = x @ self.projection

x = x / torch.norm(x, dim=-1, keepdim=True)

return x
```

In [ ]:
class TextEncoder(nn.Module):
    def __init__(self, vocab_size, width, max_seq_length, n_heads, n_layers, emb_dim):
        super().__init__()

        self.max_seq_length = max_seq_length  # Maximum length of input sequence

        self.encoder_embedding = nn.Embedding(vocab_size, width) # Embedding table

        self.positional_embedding = PositionalEmbedding(width, max_seq_length)

        self.encoder = nn.ModuleList(
            [
            TransformerEncoder(width,n_heads) for _ in range(n_layers)
            ]
        )

        # Learned projection of token's vector representation to joint embedding space
        self.projection = nn.Parameter(torch.randn(width, emb_dim))

    def forward(self, text, mask=None):
        # Obtain the token's vector representation (token<=>character)
        x = self.encoder_embedding(text)

        # Obtain the token's vector representation (token<=>character)
        x = self.positional_embedding(x)

        # Pass the tokens+PE and masks through the transformer blocks of the text encoder
        for encoder_layer in self.encoder:
            x = encoder_layer(x, mask=mask)

        # Use as encoded representation the text features produced for the EOS token
        x = x[torch.arange(text.shape[0]),torch.sub(torch.sum(mask[:,0],dim=1),1)]

        # Project encoded text features into the joint embedding space
        if self.projection is not None:
            x = x @ self.projection

        # Normalize the projected features
        x = x / torch.norm(x, dim=-1, keepdim=True)

        return x

## Image Encoder

For the image encoder, we are going to be using a vision transformer whose architecture is depicted in the next figure.

![Figure 9: Vision Transformer Model Diagram.](../fig/clip_model_01G.jpg)

When creating the image encoder, we first need to make sure that the input images can be split evenly into patches of size `patch_size` and that the dimensionality of the token's vector representation is divisible by the number of attention heads.

```python
assert img_size[0] % patch_size[0] == 0 and img_size[1] % patch_size[1] == 0, 
   "image size must be divisible by the patch size"
assert width % n_heads == 0, "width must be divisible by the number of heads"
```

We also need to calculate the maximum sequence length for the positional encoding, which will be equal to the number of patches plus one. The number of patches can be found by dividing the product of the height and width of the input image by the product of the height and width of the patch size.

```python
self.n_patches = (self.img_size[0] * self.img_size[1]) // (self.patch_size[0] * self.patch_size[1])
self.max_seq_length = self.n_patches + 1
```

The vision transformer might also be built with multiple transformer blocks. This can be achieved by creating a list of transformer blocks with `nn.ModuleList`.

```python
self.encoder = nn.ModuleList(
    [
    TransformerEncoder(width,n_heads) for _ in range(n_layers)
    ]
)
```

Before we pass the input image through the encoder layers, we first need to split the image into patches/tokens and create a sequence of linear embeddings of these patches. We are able to achieve this by using PyTorch’s `Conv2d` layers.

The `Conv2d` layer takes the input images, splits them into patches/tokens by implementing a linear projection that condenses each image patch into a single token. By setting `kernel_size` and `stride` equal to the patch size, we ensure that each token condensed a region of size equal to the patch size and there is no overlap between tokens.

```python
self.linear_project = nn.Conv2d(n_channels, width, kernel_size=patch_size, stride=patch_size)
```

In the `forward` method we pass the input image, which has shape `(BS, C, H, W)`, through the linear projection (`Conv2D`) layer to obtain image tokens. The output has a shape equal to `(BS, width, P_col, P_row)`, where `width` (`d_model` in the next figure) is the size of the token's vector representation, `P_col` is the number of patches/tokens in the vertical direction and `P_row` is the number of patches/tokens in the horizontal direction. This process is illustrated in the next figure.

```python
def forward(self, x):
    x = self.linear_project(x) # (BS, C, H, W) -> (BS, width, P_col, P_row)
```

![Figure 10: Conv2D applied on a single image. Each color represents which patch an element belongs to.](../fig/clip_model_01H.jpg)


Then, we apply the `flatten` method to rearrange the image patches into a single dimension, resulting in a tensor with shape `(BS, width, P)`, where `P=P_col*P_row`. Flattening is illustrated in the next figure.

```python
# (BS, width, P_col, P_row) -> (BS, width, P_col*P_row)
x = x.flatten(2) 
```

![Figure 11: Flatten applied to Conv2d output.](../fig/clip_model_01I.jpg)

Finally we use the `transpose` method to commute the width and patch dimensions, and obtaining a tensor of shape equal to `(BS, P, width)`. Transposing is illustrated in the next figure.

```python
x = x.transpose(-2, -1) # (BS, width, P) -> (BS, P, width)
```

![Figure 12: Transpose applied to flatten output.](../fig/clip_model_01J.jpg)

Vision transformers uses the standard approach of adding a learnable classification token to the patch embeddings in order to perform classification. This is implemented with a scalar `nn.Parameter`.

```python
self.cls_token = nn.Parameter(torch.randn(1, 1, width))
```

Each image in the batch needs to have the class token, so we use the `expand` method in order to append the class token `self.cls_token` to every image in the batch.

```python
x = torch.cat((self.cls_token.expand(x.size()[0], -1, -1),x), dim=1)
```

After adding the class tokens, we add the positional encoding to each token vector representation.

```python
x = self.positional_embedding(x)
```

With the positional encodings added, we can now pass the embeddings through the transformer layers of the image encoder.

```python
# Pass tokens+PE through the transformer layers
for encoder_layer in self.encoder:
    x = encoder_layer(x)
```

The output of the transformer layers are the learned features obtained for the class token.

```python
# Output is the features obtained for the class tokens
x = x[:, 0, :]
```

Finally, we project the image features in the joint embedding space by calculating the dot product between the features and the learned projection. The produced output is normalize dividing by the dot product norm.

```python
if self.projection is not None:
    x = x @ self.projection
x = x / torch.norm(x, dim=-1, keepdim=True)
turn x
```

In [ ]:
class ImageEncoder(nn.Module):
    def __init__(self, width, img_size, patch_size, n_channels, n_layers, n_heads, emb_dim):
        super().__init__()

        assert img_size[0] % patch_size[0] == 0 and img_size[1] % patch_size[1] == 0, \
               "image size must be divisible by the patch size"
        assert width % n_heads == 0, "width must be divisible by number of attention heads"

        self.n_patches = (img_size[0] * img_size[1]) // (patch_size[0] * patch_size[1])

        self.max_seq_length = self.n_patches + 1

        # Patch embedding
        self.linear_project = nn.Conv2d(n_channels, width, kernel_size=patch_size, stride=patch_size)

        # Class token
        self.cls_token = nn.Parameter(torch.randn(1, 1, width))

        self.positional_embedding = PositionalEmbedding(width,self.max_seq_length)

        self.encoder = nn.ModuleList([TransformerEncoder(width,n_heads) for _ in range(n_layers)])

        # Learned projection of image features to joint embedding sapce
        self.projection = nn.Parameter(torch.randn(width, emb_dim))


    def forward(self,x):
        # Create image tokens (for image patches) and arrange them in a 1D sequence
        x = self.linear_project(x)
        x = x.flatten(2).transpose(1, 2)

        # Append class token to the image tokens and add positional embedding
        x = torch.cat((self.cls_token.expand(x.size()[0], -1, -1),x), dim=1)
        x = self.positional_embedding(x)

        # Pass tokens+PE through the transformer layers
        for encoder_layer in self.encoder:
            x = encoder_layer(x)

        # Output is the features obtained for the class tokens
        x = x[:, 0, :]

        # Project encoded text features into the joint embedding space
        if self.projection is not None:
            x = x @ self.projection

        x = x / torch.norm(x, dim=-1, keepdim=True)

        return x


## CLIP Model

When it is fed with a batch of images and captions, CLIP is supposed to indicate which captions goes with which images. It does this by training the text and image encoder together to maximize the pairwise cosine similarity scores of the pairs that are supposed to match and minimizing the pairs that are not supposed to match.

To do this, we first need to get the embedded features from the image and text encoders.

```python
class CLIP(nn.Module):
   ...
   def forward(self,image,text, mask=None):
       I_e = self.image_encoder(image)
       T_e = self.text_encoder(text, mask=mask)
       ...
```

Using the embedded features, we can calculate the scaled pairwise cosine similarities by using a dot product between the embedded image features and a transposed version of the embedded text features. The cosine similarity should be maximized along the diagonal of the similarity matrix, illustrated in blue on the figure 13, which corresponds to the correct image-text pairs.

![Figure 13: Calculating cosine similarity.](../fig/clip_model_01K.jpg)<BR>
*Figure 13: Calculating cosine similarity.*

```python
logits = (I_e @ T_e.transpose(-2,-1)) * torch.exp(self.temperature)
```

This works since `I_e` and `T_e` both contain $N$ tokens, resulting in the matrix shown in the figure 13. In order to maximize the cosine similarity between related image and text, CLIP uses symmetric/contrastive loss. To calculate the contrastive loss, we first create a `labels` vector containing the indices of the text that matches with each image in the batch: 0 for pair (I0,T0), 1 for pair (I1,T1), etc. This vectors plays the role of the "true labels" on cross-entropy loss in a classification problem.

```python
# Symmetric loss function
labels = torch.arange(logits.shape[0]).to(self.device)
```

We then calculate the cross-entropy loss along the rows of the similarity matrix (`logits`) to get the loss for the images.

```python
loss_i = nn.functional.cross_entropy(logits.transpose(-2,-1), labels)
```

The loss for the text is calculated by calculating the cross-entropy loss along the columns of the similarity matrix (`logits`).

```python
loss_t = nn.functional.cross_entropy(logits, labels)
```

We get the final loss by averaging the image and text losses.

```python
loss = (loss_i + loss_t) / 2
return loss
```

In [ ]:
class CLIP(nn.Module):
    def __init__(
            self,
            emb_dim,
            vit_width,
            img_size,
            patch_size,
            n_channels,
            vit_layers,
            vit_heads,
            vocab_size,
            text_width,
            max_seq_length,
            text_heads,
            text_layers,
        ):
        super().__init__()

        self.image_encoder = ImageEncoder(
            vit_width,
            img_size,
            patch_size,
            n_channels,
            vit_layers,
            vit_heads,
            emb_dim,
        )

        self.text_encoder = TextEncoder(
            vocab_size,
            text_width,
            max_seq_length,
            text_heads,
            text_layers,
            emb_dim,
        )

        self.temperature = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))

        self.device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")


    def forward(self,image,text, mask=None):
        # Get the image embeddings
        I_e = self.image_encoder(image)
        # Get the text embeddings
        T_e = self.text_encoder(text, mask=mask)

        # Calculate the scaled cosine similarity between image and text embeddings
        logits = (I_e @ T_e.transpose(-2,-1)) * torch.exp(self.temperature)

        # Define the labels that specify the correct image-text pairs
        labels = torch.arange(logits.shape[0]).to(self.device)

        # Calculate the CE loss for images
        loss_i = nn.functional.cross_entropy(logits.transpose(-2,-1), labels)
        # Calculate the CE loss for text
        loss_t = nn.functional.cross_entropy(logits, labels)

        # Calculate the symmetric contrastive loss
        loss = (loss_i + loss_t) / 2

        return loss

## Dataset

In this notebook, we are going to use the Fashion MNIST dataset from Hugging Face. We have chosen this dataset because it is rather small and keeps training time reasonable.

```python
self.dataset = load_dataset("fashion_mnist")
```

For each entry in the dataset, we need three things: the image, the caption, and a text mask.

For each image, we only have to transform it into a tensor. In the next line of code, `self.split` specifies the train or test subset of the dataset.

```python
img = self.dataset[self.split][i]["image"]
img = self.transform(img)
```

For each caption, we have apply the tokenizer that we created before in order to get the token representation along with the mask for the tokens. In the next line of code, `self.captions[self.dataset[self.split][i]["label"]]` retrieves a caption from a dictionary given the image label.

```python
cap, mask = tokenizer(self.captions[self.dataset[self.split][i]["label"]])
```

The mask that we get from the tokenizer is a 1D tensor of size `max_seq_length`. In the text encoder, the mask is going to be applied to the attention scores which has a shape of (`max_seq_length, max_seq_length)`. Because of this, we need to expand the mask so that it is repeated for each row of the attention scores.

![Figure 14: Mask before and after expanding.](../fig/clip_model_01L.jpg)

```python
mask = mask.repeat(len(mask),1)
```

The image, caption, and mask are returned by the `__getitem__` method of the `FashionMNIST` dataset class as a dictionary.

```python
def __getitem__(self,i):
   ...
   return {"image": img, "caption": cap, "mask": mask}
```

In [ ]:
class FashionMNIST(Dataset):

    def __init__(self, train=True):
        self.dataset = load_dataset("fashion_mnist")

        self.transform = T.ToTensor()

        if train:
            self.split = "train"
        else:
            self.split = "test"

        self.captions = {
            0: "An image of a t-shirt/top",
            1: "An image of trousers",
            2: "An image of a pullover",
            3: "An image of a dress",
            4: "An image of a coat",
            5: "An image of a sandal",
            6: "An image of a shirt",
            7: "An image of a sneaker",
            8: "An image of a bag",
            9: "An image of an ankle boot"
        }


    def __len__(self):
        return self.dataset.num_rows[self.split]


    def __getitem__(self,i):
        # Provide image index 'i' from the dataset
        img = self.dataset[self.split][i]["image"]
        # Convert image to tensor
        img = self.transform(img)

        # Get caption and mask associated with image using the tokenizer
        cap, mask = tokenizer(self.captions[self.dataset[self.split][i]["label"]])

        # Replicate the mask a number of times equal the maximum sequence length
        mask = mask.repeat(len(mask),1)

        return {"image": img, "caption": cap, "mask": mask}

# Training Parameters

In [ ]:
def get_config():
    '''
    Defines the configuration for training the CLIP model.
    '''
    config = ml_collections.ConfigDict()

    config.root_dir        = "OUR_WORK_DIR"
    config.models_dir      = "models"
    config.experiment_name = "clip_fashion_mnist_01"
    config.log_interval    = 10     # Number of iterations between successive logs.
    config.train           = False  # Train the model (True) or not (False).
    config.evaluate        = True   # Evaluate the model (True) or not (False).
    config.zero_shot       = True   # Apply the model in zero-shot classification (True) or not (False).

    config.img_size        = (28,28) # Image height and width.
    config.patch_size      = (7,7)   # Size of the image patches.
    config.n_channels      = 1       # Image color channels.
    config.vit_width       = 32      # Size of the vector representation for each image token (heads*layers ????).
    config.vit_layers      = 4       # Number of transformer blocks in the image encoder transformer.
    config.vit_heads       = 8       # Number of attention heads in each image transformer block.

    config.vocab_size      = 256     # Number of different characters/codes allowed in input strings.
    config.text_width      = 32      # Size of the vector representation for each text token.
    config.max_seq_length  = 32      # Maximum number of characters an input string can have.
    config.emb_dim         = 128     # Dimensionality of the joint image-text embedding space.
    config.text_layers     = 4       # Number of transformer blocks in the text encoder transformer.
    config.text_heads      = 8       # Number of attention heads in each text transformer block.

    config.lr             = 0.0002  # Learning rate.
    config.epochs         = 100     # Number of training epochs.
    config.batch_size     = 128     # Batch size.

    config.device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
    print("Using device: ", config.device, f"({torch.cuda.get_device_name(config.device)})" \
          if torch.cuda.is_available() else "")

    return config

def print_config(config):
  '''
  Prints the configuration.
  '''
  print('Configuration parameters:')
  for name, values in config.items():
      print(f'\t{name}: {values}')

In [ ]:
# Set training configuration ..................................................
config = get_config()

# Print training configuration .................................................
print_config(config)
#print(config)

## Create the Training and Test Datasets and DataLoaders

In [ ]:
train_set    = FashionMNIST(train = True)
test_set     = FashionMNIST(train = False)

trainset_len = len(train_set)
testset_len  = len(test_set)

train_loader = DataLoader(
    train_set,
    shuffle    = True,
    batch_size = config.batch_size,
)

test_loader  = DataLoader(
    test_set,
    shuffle    = False,
    batch_size = config.batch_size,
)

In [ ]:
def print_caption(trainset, caption_id):
    cap_len = trainset[caption_id]["caption"].shape[0]
    for c in range(0,cap_len):
        ch = train_set[caption_id]["caption"][c].item()
        if c == (cap_len-1):
            print(f'{chr(ch)}')
        elif ch != BOS and ch != EOS and ch != PAD_TOKEN:
            print(f'{chr(ch)}', end='')

In [ ]:
print(f'Train dataset size: {trainset_len}')
print(f'Test  dataset size: {testset_len}')

print(f'Image shape:        {train_set[0]["image"].shape}')
print(f'Caption shape:      {train_set[0]["caption"].shape}')
print(f'Caption mask shape: {train_set[0]["mask"].shape}')
print(f'Caption #0:         ', end ='')
print_caption(train_set, 0)
print(f'Caption mask:\n{train_set[0]["mask"]}')

## Login into Weights & Bias

In [ ]:
wandb.login()

## Track metadata and hyperparameters with Weights & Bias

Define the experiment: the hyperparameters, the dataset and model name. This information will be stored in a `config` dictionary.

In [ ]:
config_wandb = config
wandb.init(
    project = 'OUR_WANDB_PROJECT_ID',
    entity  = 'OUR_WANDB_ENTITY', 
    config  = config_wandb,
    id      = config.experiment_name,
    resume  = 'allow'
    )

## Create a CLIP Model Instance

In [ ]:
file_name  = f'{config.experiment_name}.pth'
model_path = os.path.join(config.root_dir, config.models_dir, config.experiment_name)
os.makedirs(model_path, exist_ok=True)
model_path = os.path.join(model_path, file_name)

model = CLIP(
    config.emb_dim,
    config.vit_width,
    config.img_size,
    config.patch_size,
    config.n_channels,
    config.vit_layers,
    config.vit_heads,
    config.vocab_size,
    config.text_width,
    config.max_seq_length,
    config.text_heads,
    config.text_layers,
).to(config.device)

# If a model checkpoint exists load it
if os.path.isfile(model_path) == True:
    loaded_state = torch.load(model_path, map_location=config.device)
    model.load_state_dict(loaded_state['model'], strict=False)
    best_loss    = loaded_state['loss']
else:
    best_loss = np.inf

## Train the CLIP Model

In [ ]:
if config.train == True:

    optimizer = optim.Adam(model.parameters(), lr=config.lr)
    
    step         = 0
    loop_epochs  = trange(config.epochs)
    
    for epoch in loop_epochs:
    
        loop_batches = tqdm(train_loader, leave=False)
        for i, data in enumerate(loop_batches, 0):
            img  = data["image"].to(config.device)
            cap  = data["caption"].to(config.device)
            mask = data["mask"].to(config.device)
            loss = model(img, cap, mask)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
            if (step+1)%config.log_interval == 0:
                loop_batches.set_postfix_str(f' | loss: {loss.item()}')
                try:
                    # Log metrics to Weights and Biases .........................
                    wandb.log(
                        {
                        "epoch": epoch + 1,
                        "step":  step+1,
                        "loss":  loss.item(),
                        }
                    )
                except Exception as ex:
                    print(f'[ERROR] An exception of type {type(e).__name__} occurred. Arguments:\n{ex.args!r}')
            step +=1
    
        # Save the model if it performed better than the previous best instance
        print(f"\nEpoch [{epoch+1}/{config.epochs}] | Batch loss: {loss.item():.3f}")
        if loss.item() <= best_loss:
            best_loss = loss.item()
            save_state = {}
            save_state['loss']  = best_loss
            save_state['model'] = model.state_dict()
            torch.save(save_state, model_path)
            print(f'The model checkpoint was saved to {model_path}')

## Evaluate the Model on the Test Dataset

In [ ]:
if config.evaluate == True:

    # Get test dataset captions to compare images to
    text = torch.stack([tokenizer(x)[0] for x in test_set.captions.values()]).to(config.device)
    mask = torch.stack([tokenizer(x)[1] for x in test_set.captions.values()])
    # Replicate the mask for all sequence tokens
    mask = mask.repeat(1,len(mask[0])).reshape(len(mask),len(mask[0]),len(mask[0])).to(config.device)
    
    correct, total = 0,0
    with torch.no_grad():
        for data in test_loader:
            images, labels = data["image"].to(config.device), data["caption"].to(config.device)
            image_features = model.image_encoder(images)
            text_features  = model.text_encoder(text, mask=mask)
    
            image_features /= image_features.norm(dim=-1, keepdim=True)
            text_features  /= text_features.norm(dim=-1, keepdim=True)
            similarity      = (100.0 * (image_features @ text_features.T)).softmax(dim=-1)
            _, indices      = torch.max(similarity,1)
            pred            = torch.stack(
                              [tokenizer(test_set.captions[int(i)])[0] for i in indices]
                              ).to(config.device)
            correct        += int(sum(torch.sum((pred==labels),dim=1)//len(pred[0])))
            total          += len(labels)
    
    print(f'\nModel accuracy: {100 * correct // total} %')
    
    # Model accuracy: 88 %

We tested the model by getting the captions that the model was trained on and comparing it to the actual captions. When training, we used the same caption template (“An image of a(n) {class}”), so this testing stage is pretty much the same as any other image classifier. We achieved a model accuracy of 88%.

## Zero-Shot Classification

For zero-shot classification, we compare the image to just the class names. We input in the labels to compare against the image and it will return the top 5 predictions with their likelihoods. This is not a great example of applying CLIP to zero-shot classification. Using the fashion MNIST dataset makes the model easy to train, but captions are not very rich. To truly appreciate the zero-shot capabilities of CLIP, a training set where captions have multiple nouns would be more appropriate. True zero-shot detection would then allow us to detect previously unseen images.

![Figure 15: Zero-shot classification output.](../fig/clip_model_01M.jpg)

In [ ]:
import random


def generate_random_numbers(number, low, high):
    return [random.randint(low, high - 1) for _ in range(number)]


def create_image_grid(dataset, image_id, gridH=5, gridW=5):
    fig, axs = plt.subplots(gridH, gridW, figsize=(gridH*2, gridW*2))

    for i in range(gridH):
        for j in range(gridW):
            idx     = image_id[i*gridW+j]
            img     = dataset[idx]["image"][None,:]
            caption = tokenizer(dataset[idx]["caption"], encode=False, mask=test_set[idx]["mask"][0])[0]
            axs[i, j].imshow(img[0].permute(1, 2, 0)  ,cmap="gray")  
            axs[i, j].set_title(caption, fontsize=8)
            axs[i, j].axis('off')
    plt.tight_layout()
    plt.show()


def create_image_prediction_grid(dataset, class_captions, gridH=3, gridW=4):

    # Stack the 'class_captions" captions 
    text = torch.stack([tokenizer(x)[0] for x in class_captions]).to(config.device)
    # Create the attention mask to exclude non-usefull tokens
    mask = torch.stack([tokenizer(x)[1] for x in class_captions])
    # Replicate the mask for all sequence tokens
    mask = mask.repeat(1,len(mask[0])).reshape(len(mask),len(mask[0]),len(mask[0])).to(config.device)

    image_id = generate_random_numbers(GRIDH*GRIDW, 0, testset_len)
    fig, axs = plt.subplots(GRIDH, GRIDW, figsize=(GRIDW*3, GRIDH*4))
    plt.subplots_adjust(hspace=0.6)

    for i in range(GRIDH):
        for j in range(GRIDW):
            idx     = image_id[i*GRIDW+j]
            img     = test_set[idx]["image"][None,:]
            caption = tokenizer(test_set[idx]["caption"], encode=False, mask=test_set[idx]["mask"][0])[0]

            axs[i, j].imshow(img[0].permute(1, 2, 0)  ,cmap="gray")  
            
            img = img.to(config.device)
            
            with torch.no_grad():
              image_features = model.image_encoder(img)
              text_features  = model.text_encoder(text, mask=mask)
            
            # Compute the similarity matrix among input image and all possible captions
            image_features /= image_features.norm(dim=-1, keepdim=True)
            text_features  /= text_features.norm(dim=-1, keepdim=True)
            similarity      = (100.0 * image_features @ text_features.T).softmax(dim=-1)
            # Pick the top-5 similarity scores
            values, indices = similarity[0].topk(5)
         
            # Print the prediction result
            text1 = "\nTop predictions:\n"
            text2 = "".join(f"{class_captions[int(idC)]:>16s}: {100 * values[int(idV)]:.2f}%\n" for idV, idC in enumerate(indices))
            title = "True caption:\n" + caption + text1 + text2
            
            axs[i, j].set_title(title, fontsize=9)
            axs[i, j].axis('off')


In [ ]:
if config.zero_shot == True:
    '''
    # Captions to compare images to
    class_captions =[
        "t-shirt/top",
        "trousers",
        "pullover",
        "dress",
        "coat",
        "sandal",
        "shirt",
        "sneaker",
        "bag",
        "ankle boot"
    ]
    '''
    class_captions = [
        "An image of a t-shirt/top",
        "An image of trousers",
        "An image of a pullover",
        "An image of a dress",
        "An image of a coat",
        "An image of a sandal",
        "An image of a shirt",
        "An image of a sneaker",
        "An image of a bag",
        "An image of an ankle boot"
    ]
    
    # ................................................................
    # Plot a grid of images from the test set

    GRID_SIZE = 5
    # List of random image IDs to display
    image_id  = generate_random_numbers(GRID_SIZE**2, 0, testset_len)
    create_image_grid(test_set, image_id, 5, 5)

    # ................................................................
    # Plot a grid of images from the test set together with model's top-5 class predictions

    GRID_H   = 3
    GRID_W   = 4
    image_id = generate_random_numbers(GRID_H*GRID_W, 0, testset_len)
    create_image_prediction_grid(test_set, class_captions, GRID_H, GRID_W)


## Finishing the Connection to Weights & Bias

In [ ]:
# Mark the Weights and Bias run as finished
wandb.finish()